# Lora单卡训练 

In [1]:
import os
import json
import torch
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, TaskType, get_peft_model
from modelscope import snapshot_download

In [2]:
traindata_path = "../datasets/11/train_test/train0819.jsonl"
evaldata_path = "../datasets/11/train_test/eval0819.jsonl"
model_path = "../models/Qwen2-0.5B-Instruct"
output_path = "../models/Qwen2-0.5B-Instruct_fr_0819"

In [3]:
model_id = "Qwen/Qwen2-0.5B-Instruct"
print("开始从 modelscope下载模型")

snapshot_download(
    model_id=model_id,
    local_dir=model_path,
    # ModelScope 默认就是下载实体文件，不需要特别指定 symlinks 参数
)


开始从 modelscope下载模型


2026-05-23 19:37:29,774 - modelscope - INFO - Target directory already exists, skipping creation.


'../models/Qwen2-0.5B-Instruct'

In [32]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as file:
        data = [json.loads(line) for line in file]
    return data


In [41]:
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False, padding_side="left", trust_remote_code=True)
tokenizer

Qwen2Tokenizer(name_or_path='../models/Qwen2-0.5B-Instruct', vocab_size=151643, model_max_length=32768, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [6]:
tokenizer("你是谁")

{'input_ids': [105043, 100165], 'attention_mask': [1, 1]}

## 数据预处理

In [7]:
def preprocess(item, tokenizer, max_length=2048, instruction=None):
    system_message = "You are a helpful assistant."
    instruction = item["instruction"] if instruction is None else instruction
    user_message = instruction + "\n" + item["input"]
    assistant_message = json.dumps({"is_fraud": item["label"]}, ensure_ascii=False)

    message = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": assistant_message},
    ]
    # 
    full_ids = tokenizer.apply_chat_template(
        message,
        tokenize=True,
        add_generation_prompt=False,
        max_length=max_length,
        truncation=True,
    )
    prompt_ids = tokenizer.apply_chat_template(
        message[:-1],
        tokenize=True,
        add_generation_prompt=True,
        max_length=max_length,
        truncation=True,
    )
    full_ids = full_ids["input_ids"]
    prompt_ids = prompt_ids["input_ids"]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


In [8]:
def load_dataset(train_path, eval_path, tokenizer):
    train_df = load_jsonl(train_path)
    train_ds = Dataset.from_pandas(train_df)
    train_dataset = train_ds.map(
        lambda x: preprocess(x, tokenizer),
        remove_columns=train_ds.column_names,
        desc="Tokenizing train dataset",
    )

    eval_df = load_jsonl(eval_path)
    eval_ds = Dataset.from_pandas(eval_df)
    eval_dataset = eval_ds.map(
        lambda x: preprocess(x, tokenizer),
        remove_columns=eval_ds.column_names,
        desc="Tokenizing eval dataset",
    )

    return train_dataset, eval_dataset

In [9]:
train_dataset, eval_dataset = load_dataset(traindata_path, evaldata_path, tokenizer)

Tokenizing train dataset:   0%|          | 0/18787 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2348 [00:00<?, ? examples/s]

In [10]:
print(train_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 18787
})


In [11]:
print(
    f"Input IDs: {train_dataset[0]['input_ids']}\n"
    f"Attention Mask: {train_dataset[0]['attention_mask']}\n"
    f"Labels: {train_dataset[0]['labels']}"
)

Input IDs: [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 271, 100431, 99639, 37474, 105051, 108704, 11, 220, 14880, 101042, 105051, 43815, 107189, 106037, 101052, 3837, 23031, 2236, 68805, 66017, 103929, 104317, 59151, 9623, 761, 97957, 25, 830, 91233, 8, 3407, 110395, 18, 25, 10236, 236, 108, 102865, 101393, 99487, 101314, 100006, 101189, 100006, 85336, 99360, 102683, 99225, 106630, 104528, 3837, 85336, 26939, 99487, 104671, 100634, 20412, 104917, 100634, 99557, 104366, 115203, 99487, 108398, 100634, 99650, 104468, 3837, 99650, 99725, 100662, 99792, 99692, 46944, 46944, 104160, 32757, 8997, 110395, 16, 25, 58230, 109, 20412, 151645, 198, 151644, 77091, 198, 4913, 285, 761, 97957, 788, 895, 92, 151645, 198]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [12]:

tokenizer.decode(train_dataset[0]['input_ids'], skip_special_tokens=True)

'system\nYou are a helpful assistant.\nuser\n\n下面是一段对话文本, 请分析对话内容是否有诈骗风险，以json格式输出你的判断结果(is_fraud: true/false)。\n\n发言人3: 现在我所在这个哪里能够工艺能够去把屈光做得很好的，去到这个省级医院是自治区医院跟广西医科大学这个附属医院他们还可以，他们一直保持比较好的一个一个手术量。\n发言人1: 就是\nassistant\n{"is_fraud": false}\n'

In [13]:
# 
tokenizer.decode(list(filter(lambda x: x != -100, train_dataset[0]["labels"])))

'{"is_fraud": false}<|im_end|>\n'

In [14]:
device = "cuda:0"

In [15]:
def load_model(model_path, device='cuda'):
    model = AutoModelForCausalLM.from_pretrained(model_path,torch_dtype=torch.bfloat16)
    model.enable_input_require_grads() # 开启梯度检查点时，要执行该方法
    return model.to(device)

model = load_model(model_path, device)
model

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## 插入微调矩阵

使用Lora进行微调时，需要修改模型结构，这里将一个rank=8的低秩矩阵插入到模型的每个DecodeLayer层中，在训练时只学习这个低秩矩阵，原模型的参数不改变。

1. target_modules：定义了要对模型的哪些块做修改，准确来说是在具体哪些块中插入低秩矩阵。
2. r: 低秩矩阵的秩大小，值越小，模型能学习的参数越少，这里使用默认的8.
3. lora_alpha： 一个缩放比例因子，控制着模型推理过程中将LoRA参数在模型整个参数中所占的比重大小，这里也按推荐配置为r的2倍。
4. lora_dropout: 训练过程中，随机丢弃的神经元比例，目的是引入随机性来增强模型的泛化能力。

这里lora_alpha/r=2还是lora_alpha/r=4的这个比值不同所导致的原因，曾有论文实际验证过，这个比值等于2时有最好的效果，参考使用 [LoRA 微调 LLM 的实用技巧](https://www.jiqizhixin.com/articles/2023-12-04-14)。

In [16]:
def build_peft_model(model):
    config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, 
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        inference_mode=False, # 训练模式
        r=8, 
        lora_alpha=16,   
        lora_dropout=0.05
    )
    return get_peft_model(model, config)

peft_model = build_peft_model(model)
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
      

上面会发现Lora A 和 Lora B，因为是并列关系，所以
```
(base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
```
这里的A ---> B 和 base_layer 是并列的

然后这里会发现有embedding A 和 B, 这个是只有target modules 有 embedding token层的时候才会使用的

In [17]:
peft_model.print_trainable_parameters()

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


## 构建训练器

1. per_device_train_batch_size：每个设备单次运算的小批量大小，默认值未更改。
2. gradient_accumulation_steps：梯度累积的步骤数，原本是每4条数据更新一次参数，加上梯度累积=4后相当于每16条数据更新一次参数，相当于变相增加batch_size大小。
3. num_train_epochs：训练的总轮数，默认值为3，相当于所有数据训练3遍。
4. eval_strategy: 评估策略，可选有steps和epochs
5. eval_steps：训练多少步评估一次模型性能，每个batch_size为一步，此参数在eval_strategy=steps时适用。
6. save_steps：训练多少步自动保存一次模型参数。
7. learning_rate：学习率，默认值未更改。
8. load_best_model_at_end：训练结束时自动加载最佳模型
9. gradient_checkpointing：是否启用梯度检查点，启用梯度检查点可以减少kvcache对内存的占用，能节省内存。

据实际测试：对于1.5B batch_size=4的训练场景，未启用梯度检查点时会占用22G的显存，启用后能降到17G左右，效果还是很明显的。

In [19]:
def build_train_arguments(output_path):
    return TrainingArguments(
        output_dir=output_path,
        per_device_train_batch_size=4,  # 每个设备（如每个GPU）的训练批次大小
        gradient_accumulation_steps=4,  # 梯度累积的步骤数，相当于增大批次大小
        logging_steps=10,                
        num_train_epochs=3,    
        eval_strategy="steps",  
        eval_steps=10, # 设置评估的步数，与保存步数一致
        save_steps=10, # 为了快速演示，这里设置20，建议设置成100
        learning_rate=1e-4,
        save_on_each_node=True,
        load_best_model_at_end=True, # 在训练结束时加载最佳模型
        gradient_checkpointing=True  #  启用梯度检查点以节省内存
    )

## 构建训练器

1. data_collator：控制如何将原始数据合并成批(batch), DataCollatorForSeq2Seq 会自动处理输入序列的填充，使用 tokenizer 提供的填充标记（padding token）将不同长度的序列填充到相同的长度，以避免在训练过程中因序列长度不同而产生错误。
> 注：序列到序列（Seq2Seq）模型中，批量输入的多条文本数据通常具有不同的长度，而模型在进行矩阵运算时需要同一批次的数据有相同长度才能一起运算，否则会报错，所以需要指定padding=True参数来将输入序列填充到相同长度。

2. EarlyStoppingCallback：用于设置提前结束训练的回调，early_stopping_patience=3表示验证指标没有改进时经过3个评估周期后提前停止训练。
> 注：默认情况下，训练会跑满train_dataset和num_train_epochs指定的所有数据集和训练轮次，但存在一些场景（例如过拟合发生时）需要提前结束训练，此时就可以设置早停回调以免模型越训练越差，还有一个重要的点是避免浪费GPU算力成本。

In [20]:
def build_trainer(model, tokenizer, args, train_dataset, eval_dataset):
    return Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # 早停回调
    )

## 开始训练

In [21]:
trainer = build_trainer(peft_model, tokenizer, build_train_arguments(output_path), train_dataset, eval_dataset)
trainer.train()

Step,Training Loss,Validation Loss
10,0.101495,0.050531
20,0.037417,0.036241
30,0.029588,0.029150
40,0.035105,0.026822
50,0.021372,0.025740
60,0.032743,0.027633
70,0.022934,0.025971
80,0.032415,0.032931


TrainOutput(global_step=80, training_loss=0.0391335342079401, metrics={'train_runtime': 176.839, 'train_samples_per_second': 318.714, 'train_steps_per_second': 19.933, 'total_flos': 733712728332288.0, 'train_loss': 0.0391335342079401, 'epoch': 0.06812859271875665})

这里使用jupyter魔法命令%运行下 04 train_eval_package.ipynb， 把包导入进来

In [51]:
%run '04 train_eval_package.ipynb'

## 开始测评

会发现下面两个的精准度是不同的，因为他有个问题就是在train arguments 里面 load_best_model_at_end=True，这样训练结束的时候就会加载最合适的模型到peft_model里面。所以这里models/Qwen2-0.5B-Instruct_fr_0819/checkpoint-80" 最后一轮训练出来的不是最精准的

In [ ]:
testdata_path = "../datasets/11/train_test/train0819.jsonl"
evaluate_with_model(peft_model, tokenizer, testdata_path, device, debug=True)

tn：9125, fp:285, fn:3912, tp:5465
precision: 0.9504347826086956, recall: 0.5828090007465074, accuracy: 0.7766008410070794


In [52]:
checkpoint_path = "../models/Qwen2-0.5B-Instruct_fr_0819/checkpoint-80"
evaluate(model_path, checkpoint_path, evaldata_path, device, debug=True)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]





















































































































































































progress: 100%|██████████| 2348/2348 [00:17<00:00, 133.35it/s]

tn：1162, fp:3, fn:982, tp:201
precision: 0.9852941176470589, recall: 0.16990701606086223, accuracy: 0.5804940374787053


[原贴有单卡参数调教的代码，贴这里，很有启发意义](https://github.com/golfxiao/anti_fraud_sft/blob/master/7-lora%E5%8D%95%E5%8D%A1%E5%8F%82%E6%95%B0%E8%B0%83%E4%BC%98.ipynb)